# aiayn: Kaggle training → evaluation → report → preserved outputs

One unattended session (docs/KAGGLE.md §8–9, ADR-013, ADR-020):

1. **Auto-resume:** if a run named `RUN_NAME` exists in any attached input (`/kaggle/input/**`:
   a previous notebook output or the `aiayn-runs` dataset), training continues from it exactly.
2. Train (stops cleanly at the 11.5 h session budget), then evaluate the test set (BLEU, chrF).
3. Write `REPORT.md`, then **publish** the run to the private dataset `aiayn-runs`.

**Always launch with Save Version → Save & Run All (Commit).** Interactive sessions lose
`/kaggle/working` when they end. Settings: Accelerator **GPU T4**, Internet **ON**.
To publish to `aiayn-runs`, add the secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`
(*Add-ons → Secrets*). Without them the notebook output is the only copy.

In [ ]:
# 0. Parameters
GITHUB_URL = (
    "https://github.com/mohamed242006/Transformer-from-scratch-with-a-paper-reproduction.git"
)
GITHUB_BRANCH = "master"
EXPERIMENT = "configs/experiment/multi30k_tiny.yaml"
RUN_NAME = "multi30k_tiny_s1"  # stable name: used to find and resume the run
OVERRIDES = []  # e.g. ["train.seed=2"]; on resume only train.max_* / time budget may change
AUTO_RESUME = True
EVALUATE = True  # after a completed run: best beam 4, averaged beam 4, best greedy
PUBLISH = True  # upload runs/RUN_NAME to the private dataset below
RUNS_DATASET = "aiayn-runs"
WORK = "/kaggle/working"

In [ ]:
# 1. Code and dependencies (torch is never reinstalled)
import glob
import importlib.util
import os
import subprocess
import sys

REPO = f"{WORK}/aiayn"
if not os.path.exists(f"{REPO}/pyproject.toml"):
    clone = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_URL, REPO]
    subprocess.run(clone, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
needed = {"sentencepiece": "sentencepiece>=0.2.0", "sacrebleu": "sacrebleu>=2.4"}
missing = [spec for module, spec in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.insert(0, f"{REPO}/src")  # the editable install is only visible to new processes
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code: git", head)

In [ ]:
# 2. Kaggle API credentials from notebook secrets (needed only for publishing)
KAGGLE_USER = None
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
except Exception as exc:  # no secrets configured: the commit output stays the only copy
    print("publishing disabled (no KAGGLE_USERNAME/KAGGLE_KEY secrets):", type(exc).__name__)
print("publish to dataset:", bool(PUBLISH and KAGGLE_USER), f"({KAGGLE_USER}/{RUNS_DATASET})")

In [ ]:
# 3. Find a previous copy of this run among the attached inputs (auto-resume)
from aiayn.diagnostics.kaggle_runs import latest_run, run_progress

run_dir = f"{WORK}/runs/{RUN_NAME}"
previous = None
if AUTO_RESUME and not os.path.exists(run_dir):
    previous = latest_run(RUN_NAME, ["/kaggle/input"], unzip_into=f"{WORK}/_restored")
if previous:
    print(f"resuming from {previous} (update {run_progress(previous)})")
elif os.path.exists(run_dir):
    print(f"continuing {run_dir} from this session")
else:
    print("no previous copy found: starting a new run")

In [ ]:
# 4. Train. Exit code 0 = completed, 3 = stopped on the session budget (resumable).
cmd = [sys.executable, "-u", "scripts/train.py", "--run-dir", run_dir, "--prepare-data"]
if previous:
    cmd += ["--resume", str(previous)]
elif os.path.exists(f"{run_dir}/checkpoints"):
    cmd += ["--resume", run_dir]
else:
    cmd += ["--experiment", EXPERIMENT]
for item in OVERRIDES:
    cmd += ["--override", item]
print("$", " ".join(cmd))
TRAIN_EXIT = subprocess.run(cmd).returncode
print("training exit code:", TRAIN_EXIT)
assert TRAIN_EXIT in (0, 3), "training failed: see the log above"

In [ ]:
# 5. Evaluate the test set after a completed run (skipped if results already exist)
if EVALUATE and TRAIN_EXIT == 0:
    jobs = [("best", []), ("averaged", []), ("best", ["--greedy"])]
    for checkpoint, extra in jobs:
        tag = f"test.{checkpoint}.{'greedy' if extra else 'beam4'}.json"
        if os.path.exists(f"{run_dir}/eval/{tag}"):
            print("exists:", tag)
            continue
        evaluate = [sys.executable, "scripts/evaluate.py", "--run-dir", run_dir]
        subprocess.run(
            [*evaluate, "--split", "test", "--checkpoint", checkpoint, *extra], check=True
        )
elif TRAIN_EXIT == 3:
    print("budget reached: commit this version, then run it again to resume")

In [ ]:
# 6. Report (REPORT.md inside the run directory)
subprocess.run([sys.executable, "scripts/report.py", "--run-dir", run_dir], check=True)

In [ ]:
# 7. Publish the run to the private dataset so it survives this session
if PUBLISH and KAGGLE_USER:
    attached = glob.glob(f"/kaggle/input/{RUNS_DATASET}")
    cmd = [sys.executable, "scripts/kaggle_publish_runs.py", "--run", run_dir]
    cmd += ["--user", KAGGLE_USER, "--slug", RUNS_DATASET]
    cmd += ["--message", f"{RUN_NAME}: exit {TRAIN_EXIT}"]
    if attached:
        cmd += ["--previous", attached[0]]  # keep the other runs already in the dataset
    result = subprocess.run(cmd)
    print("published" if result.returncode == 0 else f"PUBLISH FAILED ({result.returncode})")
else:
    print("not published: the committed notebook output is the copy of this run")
print("next session: attach this notebook's output or the", RUNS_DATASET, "dataset, Run All")